In [23]:
import os
import sys
import pandas as pd
import pyspark
from pyspark.sql import SparkSession
import datetime as dt

# Force Spark to use Java 17
os.environ['JAVA_HOME'] = "/opt/homebrew/opt/openjdk@17"

In [ ]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('hw6') \
    .getOrCreate()

26/03/21 12:58:55 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [16]:
spark.version

'4.1.1'

In [ ]:
# !wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

In [17]:
file = spark.read.parquet('yellow_tripdata_2025-11.parquet')

In [20]:
file.repartition(4)
file.write.parquet('2025-11/')

In [ ]:
df = spark.read.parquet('2025-11/')


In [31]:
# df.columns
# type(df)
df.filter(df.tpep_pickup_datetime >= dt.datetime(2025,11,15,0,0,0)).filter(df.tpep_pickup_datetime < dt.datetime(2025,11,16,0,0,0)).count()

162604

In [37]:
df = df.withColumn('duration', df.tpep_dropoff_datetime-df.tpep_pickup_datetime)
df.sort(df.duration.desc()).take(1)

[Row(VendorID=2, tpep_pickup_datetime=datetime.datetime(2025, 11, 26, 20, 22, 12), tpep_dropoff_datetime=datetime.datetime(2025, 11, 30, 15, 1), passenger_count=1, trip_distance=121.17, RatecodeID=4, store_and_fwd_flag='N', PULocationID=265, DOLocationID=265, payment_type=2, fare_amount=887.1, extra=1.0, mta_tax=0.0, tip_amount=0.0, tolls_amount=0.0, improvement_surcharge=1.0, total_amount=889.1, congestion_surcharge=0.0, Airport_fee=0.0, cbd_congestion_fee=0.0, duration=datetime.timedelta(days=3, seconds=67128))]

In [38]:
3 * 24 + 67128 / 60 / 60

90.64666666666666

In [39]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-21 13:24:26--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 2600:9000:20e8:c000:b:20a5:b140:21, 2600:9000:20e8:ce00:b:20a5:b140:21, 2600:9000:20e8:a00:b:20a5:b140:21, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|2600:9000:20e8:c000:b:20a5:b140:21|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0.001s  

2026-03-21 13:24:26 (13.2 MB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [60]:
from pyspark.sql import types

schema = types.StructType([
    types.StructField('LocationID', types.IntegerType(), True),
    types.StructField('Borough', types.StringType(), True),
    types.StructField('Zone', types.StringType(), True),
    types.StructField('service_zone', types.StringType(), True)
])

df_zones = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv('taxi_zone_lookup.csv')

In [61]:
df_zones.columns
df_zones.schema

StructType([StructField('LocationID', IntegerType(), True), StructField('Borough', StringType(), True), StructField('Zone', StringType(), True), StructField('service_zone', StringType(), True)])

In [67]:
import pyspark.sql.functions as F

df = df.join(
    F.broadcast(df_zones), 
    df.PULocationID == df_zones.LocationID, 
    "left"
)

In [90]:
df.groupBy(df.Zone).count().sort('count').take(3)#.show()

[Row(Zone="Eltingville/Annadale/Prince's Bay", count=1),
 Row(Zone="Governor's Island/Ellis Island/Liberty Island", count=1),
 Row(Zone='Arden Heights', count=1)]

In [87]:
df_zones.filter(df_zones.LocationID == 5).show()

+----------+-------------+-------------+------------+
|LocationID|      Borough|         Zone|service_zone|
+----------+-------------+-------------+------------+
|         5|Staten Island|Arden Heights|   Boro Zone|
+----------+-------------+-------------+------------+

